In [1]:
# Naegleria fowleri Risk Detection Model
# This notebook uses machine learning to predict the risk of Naegleria fowleri 
# based on water parameters such as temperature, turbidity, and contaminant levels.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib


In [2]:
df = pd.read_csv("water_pollution_disease.csv")
df.head()


,Country,Region,Year,Water Source Type,Contaminant Level (ppm),pH Level,Turbidity (NTU),Dissolved Oxygen (mg/L),Nitrate Level (mg/L),Lead Concentration (µg/L),...,"Cholera Cases per 100,000 people","Typhoid Cases per 100,000 people","Infant Mortality Rate (per 1,000 live births)",GDP per Capita (USD),Healthcare Access Index (0-100),Urbanization Rate (%),Sanitation Coverage (% of Population),Rainfall (mm per year),Temperature (°C),Population Density (people per km²)
0,Mexico,North,2015,Lake,6.06,7.12,3.93,4.28,8.28,7.89,...,33,44,76.16,57057,96.92,84.61,63.23,2800,4.94,593
1,Brazil,West,2017,Well,5.24,7.84,4.79,3.86,15.74,14.68,...,27,8,77.30,17220,84.73,73.37,29.12,1572,16.93,234
2,Indonesia,Central,2022,Pond,0.24,6.43,0.79,3.42,36.67,9.96,...,39,50,48.45,86022,58.37,72.86,93.56,2074,21.73,57
3,Nigeria,East,2016,Well,7.91,6.71,1.96,3.12,36.92,6.77,...,33,13,95.66,31166,39.07,71.07,94.25,937,3.79,555
4,Mexico,South,2005,Well,0.12,8.16,4.22,9.15,49.35,12.51,...,31,68,58.78,25661,23.03,55.55,69.23,2295,31.44,414


In [3]:
columns_to_keep = [
    'Water Source Type', 'Contaminant Level (ppm)', 'Turbidity (NTU)',
    'Dissolved Oxygen (mg/L)', 'Nitrate Level (mg/L)', 'Temperature (°C)'
]
df_relevant = df[columns_to_keep].copy()

def infer_naegleria_risk(row):
    warm_temp = row['Temperature (°C)'] > 35
    high_contaminants = row['Contaminant Level (ppm)'] > 5
    stagnant_sources = row['Water Source Type'] in ['Pond', 'Lake', 'Well']
    return int(warm_temp and high_contaminants and stagnant_sources)

df_relevant['Naegleria_Risk'] = df_relevant.apply(infer_naegleria_risk, axis=1)
df_encoded = pd.get_dummies(df_relevant, columns=['Water Source Type'], drop_first=True)

X = df_encoded.drop('Naegleria_Risk', axis=1)
y = df_encoded['Naegleria_Risk']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled, y_train)


RandomForestClassifier(random_state=42)

In [5]:
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       579
           1       1.00      0.95      0.98        21

    accuracy                           1.00       600
   macro avg       1.00      0.98      0.99       600
weighted avg       1.00      1.00      1.00       600



In [6]:
joblib.dump(model, "naegleria_rf_model.joblib")
joblib.dump(scaler, "naegleria_scaler.joblib")


['naegleria_scaler.joblib']

In [7]:
df_encoded.to_csv("cleaned_naegleria_dataset.csv", index=False)
